# Aviation Accidents: Safety Analysis and Recommendations

**Analyst:** Atilio Barreda

## Decision frame

The client wants aircraft that, when an accident occurs, are associated with:

1. a low fraction of people aboard who are seriously or fatally injured, and
2. a low rate of total aircraft destruction.

Small and large aircraft are analyzed separately using a threshold of **20 people aboard**. Make-level comparisons require at least **20 accident records within the size subgroup**. Make/model comparisons require at least **10 records**, matching the assignment's robustness requirement.

This is an accident-outcome analysis, not an exposure-adjusted accident-rate study. The dataset does not contain total flights or flight hours by aircraft type.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

images_dir = Path("images")
images_dir.mkdir(parents=True, exist_ok=True)


## Exploratory Data Analysis

Load the cleaned data generated by the cleaning notebook.

In [ ]:
cleaned_candidates = [
    Path("data/AviationData_cleaned.csv"),
    Path("AviationData_cleaned.csv"),
    Path("../data/AviationData_cleaned.csv"),
]

cleaned_data_path = next(
    (path for path in cleaned_candidates if path.exists()),
    None,
)

if cleaned_data_path is None:
    raise FileNotFoundError(
        "AviationData_cleaned.csv was not found. Run "
        "Aviation_Accidents_Cleaning.ipynb first."
    )

df = pd.read_csv(cleaned_data_path, low_memory=False)

df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")
df["Destroyed"] = pd.to_numeric(df["Destroyed"], errors="coerce")
df["People.Aboard"] = pd.to_numeric(df["People.Aboard"], errors="coerce")
df["Fatal.Serious.Injury.Fraction"] = pd.to_numeric(
    df["Fatal.Serious.Injury.Fraction"], errors="coerce"
)

print(f"Loaded: {cleaned_data_path.resolve()}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
display(df.head())
display(
    df[
        [
            "People.Aboard",
            "Fatal.Serious.Injury.Fraction",
            "Destroyed",
        ]
    ].describe()
)

## Explore safety metrics across models and makes

An accident is assigned to the **small-aircraft** group when `People.Aboard <= 20` and to the **large-aircraft** group when `People.Aboard > 20`. Records without a known positive number aboard or without a calculable injury fraction are excluded from injury-rate comparisons.

In [ ]:
analysis_df = df[
    df["People.Aboard"].gt(0)
    & df["Fatal.Serious.Injury.Fraction"].notna()
    & df["Destroyed"].notna()
].copy()

small_planes = analysis_df[analysis_df["People.Aboard"] <= 20].copy()
large_planes = analysis_df[analysis_df["People.Aboard"] > 20].copy()

print(f"Usable accident records: {len(analysis_df):,}")
print(f"Small-aircraft records: {len(small_planes):,}")
print(f"Large-aircraft records: {len(large_planes):,}")

#### Analyze manufacturers

For each manufacturer, the table reports:

- accident-record count
- mean and median fatal/serious injury fraction
- 95% normal-approximation confidence interval for the mean injury fraction
- aircraft-destruction rate and its 95% confidence interval
- total known people aboard

Only size-specific manufacturer groups with at least 20 accidents are compared.

In [ ]:
def safety_summary(data, group_column, min_records):
    grouped = data.groupby(group_column, dropna=False)

    summary = grouped.agg(
        accidents=(group_column, "size"),
        people_aboard=("People.Aboard", "sum"),
        mean_injury_fraction=("Fatal.Serious.Injury.Fraction", "mean"),
        median_injury_fraction=("Fatal.Serious.Injury.Fraction", "median"),
        injury_sd=("Fatal.Serious.Injury.Fraction", "std"),
        destruction_rate=("Destroyed", "mean"),
        destruction_sd=("Destroyed", "std"),
    )

    summary = summary[summary["accidents"] >= min_records].copy()

    summary["injury_se"] = (
        summary["injury_sd"] / np.sqrt(summary["accidents"])
    )
    summary["injury_ci95_low"] = (
        summary["mean_injury_fraction"] - 1.96 * summary["injury_se"]
    ).clip(lower=0)
    summary["injury_ci95_high"] = (
        summary["mean_injury_fraction"] + 1.96 * summary["injury_se"]
    ).clip(upper=1)

    summary["destruction_se"] = (
        summary["destruction_sd"] / np.sqrt(summary["accidents"])
    )
    summary["destruction_ci95_low"] = (
        summary["destruction_rate"] - 1.96 * summary["destruction_se"]
    ).clip(lower=0)
    summary["destruction_ci95_high"] = (
        summary["destruction_rate"] + 1.96 * summary["destruction_se"]
    ).clip(upper=1)

    return summary.sort_values(
        ["mean_injury_fraction", "destruction_rate", "accidents"],
        ascending=[True, True, False],
    )

small_make_summary = safety_summary(small_planes, "Make", min_records=20)
large_make_summary = safety_summary(large_planes, "Make", min_records=20)

small_make_lowest_15 = small_make_summary.head(15)
large_make_lowest_15 = large_make_summary.head(15)

print("Small-aircraft manufacturers: lowest mean injury fractions")
display(small_make_lowest_15)

print("Large-aircraft manufacturers: lowest mean injury fractions")
display(large_make_lowest_15)

def plot_mean_with_ci(summary, title, filename):
    if summary.empty:
        print(f"No groups met the sample threshold for: {title}")
        return

    plot_data = summary.sort_values("mean_injury_fraction")
    lower_error = (
        plot_data["mean_injury_fraction"] - plot_data["injury_ci95_low"]
    )
    upper_error = (
        plot_data["injury_ci95_high"] - plot_data["mean_injury_fraction"]
    )

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar(
        plot_data.index.astype(str),
        plot_data["mean_injury_fraction"],
        yerr=np.vstack([lower_error, upper_error]),
        capsize=3,
    )
    ax.set_title(title)
    ax.set_xlabel("Manufacturer")
    ax.set_ylabel("Mean fatal/serious injury fraction")
    ax.tick_params(axis="x", rotation=60)
    fig.tight_layout()
    fig.savefig(images_dir / filename, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_mean_with_ci(
    small_make_lowest_15,
    "Small aircraft: 15 manufacturers with lowest mean injury fraction",
    "small_make_injury.png",
)
plot_mean_with_ci(
    large_make_lowest_15,
    "Large aircraft: 15 manufacturers with lowest mean injury fraction",
    "large_make_injury.png",
)

**Distribution of injury rates: small-aircraft manufacturers**

A violin plot reveals whether a low mean is representative of the full distribution or is driven by a small number of unusually safe outcomes.

In [ ]:
small_make_top_10_names = small_make_summary.head(10).index
small_make_distribution = small_planes[
    small_planes["Make"].isin(small_make_top_10_names)
].copy()

if not small_make_distribution.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.violinplot(
        data=small_make_distribution,
        x="Make",
        y="Fatal.Serious.Injury.Fraction",
        cut=0,
        inner="quartile",
        ax=ax,
    )
    ax.set_title(
        "Small aircraft: injury-fraction distributions for the 10 lowest-mean makes"
    )
    ax.set_xlabel("Manufacturer")
    ax.set_ylabel("Fatal/serious injury fraction")
    ax.tick_params(axis="x", rotation=60)
    fig.tight_layout()
    fig.savefig(images_dir / "small_make_distribution.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print("No small-aircraft manufacturer met the sample threshold.")

**Distribution of injury rates: large-aircraft manufacturers**

A strip plot shows individual accident outcomes and makes sample density visible rather than relying only on group averages.

In [ ]:
large_make_top_10_names = large_make_summary.head(10).index
large_make_distribution = large_planes[
    large_planes["Make"].isin(large_make_top_10_names)
].copy()

if not large_make_distribution.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.stripplot(
        data=large_make_distribution,
        x="Make",
        y="Fatal.Serious.Injury.Fraction",
        jitter=0.25,
        alpha=0.45,
        ax=ax,
    )
    ax.set_title(
        "Large aircraft: accident-level injury fractions for the 10 lowest-mean makes"
    )
    ax.set_xlabel("Manufacturer")
    ax.set_ylabel("Fatal/serious injury fraction")
    ax.tick_params(axis="x", rotation=60)
    fig.tight_layout()
    fig.savefig(images_dir / "large_make_distribution.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print("No large-aircraft manufacturer met the sample threshold.")

**Evaluate aircraft destruction by manufacturer**

The destruction-rate tables use the same minimum sample threshold. A lower value means a smaller proportion of recorded accidents ended in total aircraft destruction.

In [ ]:
small_destroy_lowest_15 = small_make_summary.sort_values(
    ["destruction_rate", "mean_injury_fraction", "accidents"],
    ascending=[True, True, False],
).head(15)

large_destroy_lowest_15 = large_make_summary.sort_values(
    ["destruction_rate", "mean_injury_fraction", "accidents"],
    ascending=[True, True, False],
).head(15)

print("Small-aircraft manufacturers: lowest destruction rates")
display(small_destroy_lowest_15)

print("Large-aircraft manufacturers: lowest destruction rates")
display(large_destroy_lowest_15)

def plot_destruction_rate(summary, title, filename):
    if summary.empty:
        print(f"No groups met the sample threshold for: {title}")
        return

    plot_data = summary.sort_values("destruction_rate")
    lower_error = (
        plot_data["destruction_rate"] - plot_data["destruction_ci95_low"]
    )
    upper_error = (
        plot_data["destruction_ci95_high"] - plot_data["destruction_rate"]
    )

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar(
        plot_data.index.astype(str),
        plot_data["destruction_rate"],
        yerr=np.vstack([lower_error, upper_error]),
        capsize=3,
    )
    ax.set_title(title)
    ax.set_xlabel("Manufacturer")
    ax.set_ylabel("Aircraft destruction rate")
    ax.tick_params(axis="x", rotation=60)
    fig.tight_layout()
    fig.savefig(images_dir / filename, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_destruction_rate(
    small_destroy_lowest_15,
    "Small aircraft: 15 manufacturers with lowest destruction rates",
    "small_make_destruction.png",
)
plot_destruction_rate(
    large_destroy_lowest_15,
    "Large aircraft: 15 manufacturers with lowest destruction rates",
    "large_make_destruction.png",
)

#### Manufacturer recommendations

A manufacturer should not be recommended on a single metric. The composite table below ranks each eligible manufacturer on both mean injury fraction and destruction rate, with equal weight. Lower scores are better. Accident count and confidence intervals remain visible so the decision-maker can distinguish a stable estimate from a fragile one.

The first rows of `small_make_recommendations` and `large_make_recommendations` are the recommended manufacturers for each size segment.

In [ ]:
def ranked_recommendations(summary, limit=5):
    ranked = summary.copy()
    ranked["injury_rank"] = ranked["mean_injury_fraction"].rank(
        method="min", ascending=True
    )
    ranked["destruction_rank"] = ranked["destruction_rate"].rank(
        method="min", ascending=True
    )
    ranked["composite_safety_rank"] = (
        ranked["injury_rank"] + ranked["destruction_rank"]
    ) / 2

    return ranked.sort_values(
        [
            "composite_safety_rank",
            "mean_injury_fraction",
            "destruction_rate",
            "accidents",
        ],
        ascending=[True, True, True, False],
    ).head(limit)

small_make_recommendations = ranked_recommendations(
    small_make_summary, limit=5
)
large_make_recommendations = ranked_recommendations(
    large_make_summary, limit=5
)

print("RECOMMENDED SMALL-AIRCRAFT MANUFACTURERS")
display(small_make_recommendations)

print("RECOMMENDED LARGE-AIRCRAFT MANUFACTURERS")
display(large_make_recommendations)

### Analyze specific plane types

A make/model is eligible only when it has at least **10 accident records within its size segment**. The same two primary safety outcomes and confidence intervals are calculated for each `Plane.Type`.

**Larger plane types**

In [ ]:
large_type_summary = safety_summary(
    large_planes, "Plane.Type", min_records=10
)
large_type_recommendations = ranked_recommendations(
    large_type_summary, limit=10
)

print("Recommended large-aircraft make/model combinations")
display(large_type_recommendations)

if not large_type_recommendations.empty:
    large_type_plot = large_type_recommendations.sort_values(
        "mean_injury_fraction"
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(
        large_type_plot.index.astype(str),
        large_type_plot["mean_injury_fraction"],
    )
    ax.set_title(
        "Large plane types: recommended models by mean injury fraction"
    )
    ax.set_xlabel("Make and model")
    ax.set_ylabel("Mean fatal/serious injury fraction")
    ax.tick_params(axis="x", rotation=70)
    fig.tight_layout()
    fig.savefig(images_dir / "large_model_injury.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    large_type_names = large_type_recommendations.index
    large_type_distribution = large_planes[
        large_planes["Plane.Type"].isin(large_type_names)
    ]

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.stripplot(
        data=large_type_distribution,
        x="Plane.Type",
        y="Fatal.Serious.Injury.Fraction",
        jitter=0.25,
        alpha=0.45,
        ax=ax,
    )
    ax.set_title(
        "Large plane types: accident-level injury-fraction distributions"
    )
    ax.set_xlabel("Make and model")
    ax.set_ylabel("Fatal/serious injury fraction")
    ax.tick_params(axis="x", rotation=70)
    fig.tight_layout()
    plt.show()
    plt.close(fig)
else:
    print("No large plane type met the 10-record threshold.")

**Smaller plane types**

In [ ]:
small_type_summary = safety_summary(
    small_planes, "Plane.Type", min_records=10
)
small_type_recommendations = ranked_recommendations(
    small_type_summary, limit=10
)

print("Recommended small-aircraft make/model combinations")
display(small_type_recommendations)

if not small_type_recommendations.empty:
    small_type_plot = small_type_recommendations.sort_values(
        "mean_injury_fraction"
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(
        small_type_plot.index.astype(str),
        small_type_plot["mean_injury_fraction"],
    )
    ax.set_title(
        "Small plane types: recommended models by mean injury fraction"
    )
    ax.set_xlabel("Make and model")
    ax.set_ylabel("Mean fatal/serious injury fraction")
    ax.tick_params(axis="x", rotation=70)
    fig.tight_layout()
    fig.savefig(images_dir / "small_model_injury.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    small_type_names = small_type_recommendations.index
    small_type_distribution = small_planes[
        small_planes["Plane.Type"].isin(small_type_names)
    ]

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.violinplot(
        data=small_type_distribution,
        x="Plane.Type",
        y="Fatal.Serious.Injury.Fraction",
        cut=0,
        inner="quartile",
        ax=ax,
    )
    ax.set_title(
        "Small plane types: injury-fraction distributions"
    )
    ax.set_xlabel("Make and model")
    ax.set_ylabel("Fatal/serious injury fraction")
    ax.tick_params(axis="x", rotation=70)
    fig.tight_layout()
    plt.show()
    plt.close(fig)
else:
    print("No small plane type met the 10-record threshold.")

### Discussion of specific airplane types

The recommended model tables jointly favor low injury severity and low aircraft destruction while enforcing a minimum sample. The recommendation is therefore the **top five eligible rows** in each segment, not simply the model with the single lowest observed mean.

Operational use should also review confidence intervals, fleet mission, aircraft age, maintenance environment, and exposure data. A model can have favorable accident outcomes yet still have an unknown accident rate per flight hour.

In [ ]:
def print_recommendation_list(label, recommendation_table):
    print(label)
    if recommendation_table.empty:
        print("  No model met the sample requirement.")
        return

    for plane_type, row in recommendation_table.head(5).iterrows():
        print(
            f"  {plane_type}: "
            f"{int(row['accidents'])} accidents, "
            f"mean injury fraction={row['mean_injury_fraction']:.3f}, "
            f"destruction rate={row['destruction_rate']:.3f}"
        )

print_recommendation_list(
    "SMALL-AIRCRAFT MODEL RECOMMENDATIONS",
    small_type_recommendations,
)
print()
print_recommendation_list(
    "LARGE-AIRCRAFT MODEL RECOMMENDATIONS",
    large_type_recommendations,
)

### Explore other variables associated with accident severity

Two factors are analyzed:

1. **Weather condition**
2. **Broad phase of flight**

Groups with fewer than 100 usable accident records are excluded from these factor comparisons. These are observational associations and should not be interpreted as causal effects.

In [ ]:
def factor_summary(data, factor_column, min_records=100):
    if factor_column not in data.columns:
        return pd.DataFrame()

    usable = data[
        data[factor_column].notna()
        & data[factor_column].ne("UNKNOWN")
    ].copy()

    summary = (
        usable.groupby(factor_column)
        .agg(
            accidents=(factor_column, "size"),
            mean_injury_fraction=(
                "Fatal.Serious.Injury.Fraction",
                "mean",
            ),
            median_injury_fraction=(
                "Fatal.Serious.Injury.Fraction",
                "median",
            ),
            destruction_rate=("Destroyed", "mean"),
            people_aboard=("People.Aboard", "sum"),
        )
    )

    return summary[
        summary["accidents"] >= min_records
    ].sort_values(
        ["mean_injury_fraction", "destruction_rate"],
        ascending=[False, False],
    )

weather_summary = factor_summary(
    analysis_df, "Weather.Condition", min_records=100
)
phase_summary = factor_summary(
    analysis_df, "Broad.phase.of.flight", min_records=100
)

print("WEATHER CONDITION SUMMARY")
display(weather_summary)

if not weather_summary.empty:
    weather_plot = weather_summary.sort_values(
        "mean_injury_fraction", ascending=False
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(
        weather_plot.index.astype(str),
        weather_plot["mean_injury_fraction"],
    )
    ax.set_title("Fatal/serious injury fraction by weather condition")
    ax.set_xlabel("Weather condition")
    ax.set_ylabel("Mean fatal/serious injury fraction")
    fig.tight_layout()
    fig.savefig(images_dir / "weather_injury.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(
        weather_plot.index.astype(str),
        weather_plot["destruction_rate"],
    )
    ax.set_title("Aircraft destruction rate by weather condition")
    ax.set_xlabel("Weather condition")
    ax.set_ylabel("Destruction rate")
    fig.tight_layout()
    fig.savefig(images_dir / "weather_destruction.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("BROAD PHASE OF FLIGHT SUMMARY")
display(phase_summary)

if not phase_summary.empty:
    phase_plot = phase_summary.head(12).sort_values(
        "mean_injury_fraction", ascending=False
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(
        phase_plot.index.astype(str),
        phase_plot["mean_injury_fraction"],
    )
    ax.set_title(
        "Phases of flight with the highest mean injury fractions"
    )
    ax.set_xlabel("Broad phase of flight")
    ax.set_ylabel("Mean fatal/serious injury fraction")
    ax.tick_params(axis="x", rotation=60)
    fig.tight_layout()
    fig.savefig(images_dir / "phase_injury.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(
        phase_plot.index.astype(str),
        phase_plot["destruction_rate"],
    )
    ax.set_title(
        "Aircraft destruction rate for the same phases of flight"
    )
    ax.set_xlabel("Broad phase of flight")
    ax.set_ylabel("Destruction rate")
    ax.tick_params(axis="x", rotation=60)
    fig.tight_layout()
    fig.savefig(images_dir / "phase_destruction.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("\nDATA-DRIVEN INTERPRETATION")
if not weather_summary.empty:
    highest_weather = weather_summary.index[0]
    print(
        f"- {highest_weather} has the highest mean injury fraction "
        "among weather categories meeting the sample threshold."
    )
if not phase_summary.empty:
    highest_phase = phase_summary.index[0]
    print(
        f"- {highest_phase} has the highest mean injury fraction "
        "among flight phases meeting the sample threshold."
    )

print(
    "- These relationships are associative. Aircraft mix, mission, "
    "occupancy, reporting, and accident circumstances may confound them."
)

## Executive conclusion

The insurer should use the generated recommendation tables rather than a hard-coded brand list:

- `small_make_recommendations`
- `large_make_recommendations`
- `small_type_recommendations`
- `large_type_recommendations`

Each table requires adequate sample size and combines injury severity with aircraft destruction. Weather and phase of flight are additional underwriting and operational factors because their group-level outcomes differ materially in the generated summaries.

### Limitations

- The dataset includes accidents, not every flight.
- The results estimate severity **given an accident**, not accident frequency per flight hour.
- Occupant counts and other fields are incomplete for some records.
- Manufacturer and model labels remain imperfect despite normalization.
- Historical aircraft technology, maintenance, operations, and reporting practices changed over the study period.